# 🫁 ChestX Triage Using Transfer Learning

## 📘 Project Overview

This project uses **Transfer Learning** to classify public chest X-ray images and translate the model output into a simple review workflow.

The project focuses on three classes:

- **No Finding**
- **Nodule**
- **Effusion**

The goal is not to replace a clinician. The goal is to show how a computer vision model can support triage by surfacing the predicted finding, confidence score, and review priority.

> **Disclaimer:** This notebook is an educational portfolio project and is not intended for diagnostic use.


## 🔍 Understanding Transfer Learning

Transfer Learning allows us to reuse the convolutional layers of a model trained on a large image dataset, such as ImageNet, and apply that knowledge to a new image classification task.

In CNNs, convolutional layers extract visual features such as edges, gradients, textures, and shapes. These features are often useful across different image tasks.

For this project, a pre-trained CNN backbone is used as a feature extractor, and custom dense layers are added for chest X-ray classification.


## 🧰 Tools and Libraries Used

| Library | Purpose |
|---|---|
| Pandas | Metadata handling and data analysis |
| NumPy | Numerical computations |
| Matplotlib | Data visualization |
| OpenCV | Image processing and resizing |
| scikit-learn | Data splitting, preprocessing, and model evaluation |
| TensorFlow / Keras | Transfer learning and model development |
| KaggleHub | Downloading the public Kaggle dataset |


In [ ]:
# If running in Google Colab, uncomment this line if needed:
# !pip install kagglehub opencv-python tensorflow scikit-learn -q

import os
from pathlib import Path
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 📦 Dataset Information

This notebook uses the public **NIH Chest X-ray Dataset** available on Kaggle.

The full dataset contains 112,120 chest X-ray images from 30,805 patients. Each image includes one or more finding labels.

For this portfolio build, I use a smaller balanced subset of three classes:

- No Finding
- Nodule
- Effusion

This keeps the notebook easier to run in Colab while still demonstrating the full workflow.


In [ ]:
# Download public dataset from Kaggle using KaggleHub.
# This dataset is large, so the download may take time.

import kagglehub

dataset_path = kagglehub.dataset_download("nih-chest-xrays/data")
dataset_path = Path(dataset_path)

print("Dataset downloaded to:", dataset_path)
print("Top-level files/folders:")
print(list(dataset_path.iterdir())[:10])


In [ ]:
# Locate metadata CSV and image files.

metadata_files = list(dataset_path.rglob("Data_Entry_2017.csv"))
if not metadata_files:
    metadata_files = list(dataset_path.rglob("*.csv"))

metadata_path = metadata_files[0]
print("Metadata path:", metadata_path)

df = pd.read_csv(metadata_path)
df.head()


In [ ]:
df.columns


In [ ]:
# Basic label distribution

label_col = "Finding Labels"
df[label_col].value_counts().head(15)


## 🧠 Data Preparation

The NIH dataset is multi-label, meaning some images have multiple labels separated by `|`.

To keep this project clean and close to a standard image-classification portfolio project, I filter to single-label images for:

- No Finding
- Nodule
- Effusion


In [ ]:
TARGET_CLASSES = ["No Finding", "Nodule", "Effusion"]
SAMPLE_PER_CLASS = 300

single_label_df = df[df[label_col].isin(TARGET_CLASSES)].copy()

balanced_df = (
    single_label_df
    .groupby(label_col, group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_CLASS), random_state=SEED))
    .reset_index(drop=True)
)

balanced_df[label_col].value_counts()


In [ ]:
# Build a lookup of image filename -> full path.
# Kaggle's NIH dataset stores images across multiple image folders.

image_paths = {}
for img_path in dataset_path.rglob("*.png"):
    image_paths[img_path.name] = img_path

print("Number of image paths found:", len(image_paths))

balanced_df["image_path"] = balanced_df["Image Index"].map(image_paths)
balanced_df = balanced_df.dropna(subset=["image_path"]).reset_index(drop=True)

balanced_df[["Image Index", "Finding Labels", "image_path"]].head()


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.0001

def load_and_preprocess_image(path, img_size=IMG_SIZE):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read image: {path}")

    image = cv2.resize(image, (img_size, img_size))
    image = np.stack([image, image, image], axis=-1)
    image = image.astype("float32") / 255.0
    return image


In [ ]:
# Visualize sample images

sample_df = balanced_df.sample(6, random_state=SEED)

plt.figure(figsize=(12, 6))
for i, (_, row) in enumerate(sample_df.iterrows(), start=1):
    img = load_and_preprocess_image(row["image_path"])
    plt.subplot(2, 3, i)
    plt.imshow(img, cmap="gray")
    plt.title(row[label_col])
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Encode labels

label_encoder = LabelEncoder()
balanced_df["label_encoded"] = label_encoder.fit_transform(balanced_df[label_col])

class_names = list(label_encoder.classes_)
class_names


In [ ]:
train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df["label_encoded"],
    random_state=SEED
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)


In [ ]:
def dataframe_to_arrays(dataframe):
    images = []
    labels = []

    for _, row in dataframe.iterrows():
        images.append(load_and_preprocess_image(row["image_path"]))
        labels.append(row["label_encoded"])

    return np.array(images), np.array(labels)

X_train, y_train = dataframe_to_arrays(train_df)
X_val, y_val = dataframe_to_arrays(val_df)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)


## 🏗️ Model Development

The model uses **MobileNetV2**, a pre-trained CNN trained on ImageNet.

The base model is frozen and used as a feature extractor. Custom dense layers are added for the three X-ray classes.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 🚂 Model Training

The model is trained on a balanced subset so the notebook can run more easily in Google Colab.


In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


In [ ]:
# Plot training history

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss")
plt.legend()

plt.tight_layout()
plt.show()


## 📊 Model Evaluation

The model is evaluated using a classification report and confusion matrix.

For a medical-review workflow, recall and false negatives are especially important because missed findings can be high-risk.


In [ ]:
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_val, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.show()


## 🖥️ Product Workflow Translation

To make this more product-oriented, the model output is translated into a simple review-priority workflow.

The model returns:

- predicted class
- confidence score
- review priority

This is the layer that turns a model into a product surface.


In [ ]:
def assign_review_priority(predicted_label, confidence):
    high_review_classes = {"Nodule", "Effusion"}

    if predicted_label in high_review_classes and confidence >= 0.75:
        return "High Priority Review"
    elif confidence < 0.80:
        return "Manual Review"
    else:
        return "Audit Sample"

def predict_case(index):
    image = X_val[index]
    actual_label = class_names[y_val[index]]

    probs = model.predict(np.expand_dims(image, axis=0), verbose=0)[0]
    pred_index = int(np.argmax(probs))
    predicted_label = class_names[pred_index]
    confidence = float(probs[pred_index])

    review_priority = assign_review_priority(predicted_label, confidence)

    return {
        "actual_label": actual_label,
        "predicted_label": predicted_label,
        "confidence": confidence,
        "review_priority": review_priority
    }

predict_case(0)


In [ ]:
# Show sample predictions as product-style outputs

for i in range(min(5, len(X_val))):
    result = predict_case(i)
    print(f"Case {i+1}")
    print("Actual:", result["actual_label"])
    print("Predicted:", result["predicted_label"])
    print("Confidence:", round(result["confidence"], 3))
    print("Review Priority:", result["review_priority"])
    print("-" * 40)


## 🧾 Product Notes

The product design decision is to avoid presenting the model as a diagnosis.

Instead, the model supports a review workflow:

1. Classify image
2. Show confidence
3. Route high-priority or low-confidence cases to human review
4. Use reviewed cases for future feedback loops

This keeps the human reviewer central and makes uncertainty visible.


## ⚠️ Limitations

- The dataset labels are weak labels extracted from reports.
- This notebook uses a small subset for portfolio purposes.
- The model is not clinically validated.
- The workflow is educational and should not be used for diagnosis.
- Real deployment would require clinical, regulatory, privacy, and safety review.


## 🚀 Future Improvements

- Train on a larger image subset
- Add multi-label classification
- Add real Grad-CAM heatmaps
- Add model calibration
- Add active learning for uncertain cases
- Add radiologist feedback simulation
- Build a full Streamlit review dashboard
